# BraTS2021 - EDA

Just exploring the data before starting on any model code. Loading the volumes, seeing what the modalities actually look like, checking the masks.

## Setup

Usual imports. nibabel for nifti loading, nilearn I might need later. Pulled in itkwidgets too but ended up not using it.

In [1]:
import pandas as pd
import numpy as np
import os

import seaborn as sns
import matplotlib.pyplot as plt

import nilearn as nil
import nibabel as nib

import itk
import itkwidgets
from ipywidgets import interact, IntSlider, interactive, ToggleButtons

## Dataset structure

1251 cases total. Each one is its own folder with 5 .nii.gz files, four modalities and the seg mask. Quick look at what that actually contains.

In [2]:
train_data_path = '../data/src_1/BraTS2021_Training_Data/'
#validation_data_path = '../data/src_1/BraTS2021_Validation_Data/'

In [ ]:
cases = sorted([d for d in os.listdir(train_data_path) if d.startswith('BraTS2021_')])
print(f"Total cases: {len(cases)}")
print(f"\nFirst few: {cases[:5]}")
print(f"\nFiles in one case ({cases[0]}):")
for f in sorted(os.listdir(os.path.join(train_data_path, cases[0]))):
    print(f"  {f}")

In [3]:
test_image_flair = nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_flair.nii.gz').get_fdata()
test_image_t1 = nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_t1.nii.gz').get_fdata()
test_image_t1ce = nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_t1ce.nii.gz').get_fdata()
test_image_t2 = nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_t2.nii.gz').get_fdata()
test_mask = nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_seg.nii.gz').get_fdata()

test_image_flair.shape, test_image_t1.shape, test_image_t1ce.shape, test_image_t2.shape,test_mask.shape

((240, 240, 155),
 (240, 240, 155),
 (240, 240, 155),
 (240, 240, 155),
 (240, 240, 155))

All (240, 240, 155), BraTS preprocesses everything to the same shape and spacing. 1mm isotropic. A full 4-channel float32 volume is around 143MB so fitting the whole thing on the GPU at once is not happening. Patch-based training.

In [4]:
# fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1,5, figsize = (20, 10))
# slice_w = 25
# ax1.imshow(test_image_flair[:,:,test_image_flair.shape[0]//2-slice_w], cmap = 'gray')
# ax1.set_title('Image flair')
# ax2.imshow(test_image_t1[:,:,test_image_t1.shape[0]//2-slice_w], cmap = 'gray')
# ax2.set_title('Image t1')
# ax3.imshow(test_image_t1ce[:,:,test_image_t1ce.shape[0]//2-slice_w], cmap = 'gray')
# ax3.set_title('Image t1ce')
# ax4.imshow(test_image_t2[:,:,test_image_t2.shape[0]//2-slice_w], cmap = 'gray')
# ax4.set_title('Image t2')
# ax5.imshow(test_mask[:,:,test_mask.shape[0]//2-slice_w])
# ax5.set_title('Mask')

Static plot is fine for a quick check. Going to build a slider so I can actually scroll through properly.

In [5]:
%matplotlib widget

from ipywidgets import IntSlider, Button, HBox, VBox, Layout, IntText, jslink
from IPython.display import display

def create_and_display_viewer():
    plt.ioff()  #Turn off interactive mode to prevent double display of the plots
    
    fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(1, 5, figsize=(20, 6))
    
    initial_layer = test_mask.shape[2] // 2

    # Plot the initial data and store the returned image objects
    im1 = ax1.imshow(test_image_flair[:, :, initial_layer], cmap='gray')
    im2 = ax2.imshow(test_image_t1[:, :, initial_layer], cmap='gray')
    im3 = ax3.imshow(test_image_t1ce[:, :, initial_layer], cmap='gray')
    im4 = ax4.imshow(test_image_t2[:, :, initial_layer], cmap='gray')
    im5 = ax5.imshow(test_mask[:, :, initial_layer])
    
    ax1.set_title('Image flair'); ax1.axis('off')
    ax2.set_title('Image t1'); ax2.axis('off')
    ax3.set_title('Image t1ce'); ax3.axis('off')
    ax4.set_title('Image t2'); ax4.axis('off')
    ax5.set_title('Mask'); ax5.axis('off')
    fig_title = fig.suptitle(f'Showing Slice: {initial_layer}', y=0.9)
    
    layer_slider = IntSlider(
        min=0,
        max=test_mask.shape[2] - 1,
        step=1,
        value=initial_layer,
        description='Select Layer:',
        layout=Layout(width='50%'),
        readout=False 
    )
    plus_button = Button(description='+', layout=Layout(width='40px'))
    minus_button = Button(description='-', layout=Layout(width='40px'))
    
    number_input = IntText(
        value=initial_layer,
        layout=Layout(width='60px')
    )

    jslink((layer_slider, 'value'), (number_input, 'value'))
    
    def update_plot(change):
        layer = change['new']
        im1.set_data(test_image_flair[:, :, layer])
        im2.set_data(test_image_t1[:, :, layer])
        im3.set_data(test_image_t1ce[:, :, layer])
        im4.set_data(test_image_t2[:, :, layer])
        im5.set_data(test_mask[:, :, layer])
        fig_title.set_text(f'Showing Slice: {layer}')
        fig.canvas.draw_idle()
    

    def increment_layer(b):
        if layer_slider.value < layer_slider.max:
            layer_slider.value += 1
    def decrement_layer(b):
        if layer_slider.value > layer_slider.min:
            layer_slider.value -= 1
    
    layer_slider.observe(update_plot, names='value')
    plus_button.on_click(increment_layer)
    minus_button.on_click(decrement_layer)
    controls = HBox([layer_slider, minus_button, number_input, plus_button], 
                    layout=Layout(justify_content='center', align_items='center'))
    ui = VBox([controls, fig.canvas])
    
    plt.ion()  
    display(ui)


create_and_display_viewer()

Notes from scrolling through a few cases:
- tumour only shows up in maybe 30-50 slices out of 155, pretty localised
- FLAIR shows the edema ring most clearly
- T1CE is the best for seeing the enhancing tumour boundary
- necrotic core looks dark on T1CE
- quite a lot of variation between patients, some tumours are tiny, some are much bigger and irregular

## Label distribution

From looking at the slices the tumour is clearly tiny relative to the whole brain volume. Want to get the actual numbers, matters for setting up the sampler.

In [6]:
def analyze_segmentation_masks(
        patient_ids = ["00002", "00005", "00008"]
):
    # Load several examples
    patient_ids = ["00002", "00005", "00008"]
    mask_values = []
    
    for patient_id in patient_ids:
        mask = nib.load(f"{train_data_path}/BraTS2021_{patient_id}/BraTS2021_{patient_id}_seg.nii.gz").get_fdata()
        unique_values = np.unique(mask)
        mask_values.append(unique_values)
        print(f"Patient {patient_id} mask values: {unique_values}")
        
        # Count voxels per class
        for val in unique_values:
            if val > 0:  # Skip background
                count = np.sum(mask == val)
                print(f"  Class {val}: {count} voxels ({count/mask.size*100:.2f}% of volume)")
    
    return mask_values

mask_values = analyze_segmentation_masks()

Patient 00002 mask values: [0. 1. 2. 4.]
  Class 1.0: 11248 voxels (0.13% of volume)
  Class 2.0: 155695 voxels (1.74% of volume)
  Class 4.0: 23651 voxels (0.26% of volume)
Patient 00005 mask values: [0. 1. 2. 4.]
  Class 1.0: 24187 voxels (0.27% of volume)
  Class 2.0: 76742 voxels (0.86% of volume)
  Class 4.0: 23854 voxels (0.27% of volume)
Patient 00008 mask values: [0. 1. 2. 4.]
  Class 1.0: 16 voxels (0.00% of volume)
  Class 2.0: 18135 voxels (0.20% of volume)
  Class 4.0: 311 voxels (0.00% of volume)


Under 3% tumour voxels across all three cases. ED is the biggest chunk, NCR and ET are tiny. Patient 00008 barely has anything.

Random patch sampling won't work here, most patches would just be background. Need foreground-biased sampling, MONAI has RandCropByPosNegLabeld for this.

Also no label 3 anywhere. Labels go 0, 1, 2, 4.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
patient_ids_plot = ["00002", "00005", "00008"]
class_names = {1: "NCR", 2: "ED", 4: "ET"}
colors = ["#c0392b", "#e67e22", "#2980b9"]

for ax, pid in zip(axes, patient_ids_plot):
    mask = nib.load(f"{train_data_path}/BraTS2021_{pid}/BraTS2021_{pid}_seg.nii.gz").get_fdata()
    total = mask.size
    counts = {name: np.sum(mask == cls) / total * 100 for cls, name in class_names.items()}
    bars = ax.bar(counts.keys(), counts.values(), color=colors)
    ax.set_title(f"Patient {pid}")
    ax.set_ylabel("% of total volume")
    ax.set_ylim(0, 2.5)
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
                f"{val:.2f}%", ha="center", va="bottom", fontsize=9)

plt.suptitle("Tumour label distribution (% of total volume)", y=1.02)
plt.tight_layout()
plt.show()

In [7]:
print(nib.load(train_data_path + 'BraTS2021_00002/BraTS2021_00002_seg.nii.gz').header)

<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : np.bytes_(b'')
db_name         : np.bytes_(b'')
extents         : 0
session_error   : 0
regular         : np.bytes_(b'r')
dim_info        : 0
dim             : [  3 240 240 155   1   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : uint8
bitpix          : 8
slice_start     : 0
pixdim          : [1. 1. 1. 1. 0. 0. 0. 0.]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 2
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 0
glmin           : 0
descrip         : np.bytes_(b'')
aux_file        : np.bytes_(b'')
qform_code      : scanner
sform_code      : scanner
quatern_b       : 0.0
quatern_c       : 0.0
quatern_d       : 1.0
qoffset_x       : -0.0
qoffset_y       : 239.0
qoffset_z       : 0.0

pixdim is [1, 1, 1] so 1mm isotropic as expected. srow_y is flipped, standard for brain MRI. sform and qform both scanner coords. Nothing unexpected.

In [8]:
import shutil
import gif_your_nifti.core as gif2nif

In [9]:
# shutil.copy2(train_data_path)
gif2nif.write_gif_pseudocolor(train_data_path + 'BraTS2021_00002/BraTS2021_00002_flair.nii.gz')

## Hardware

Checking what I'm working with before thinking about architecture and batch sizes.

In [10]:
import os, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
torch.backends.cudnn.benchmark = True  # autotune best conv kernels after first batch


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}, PyTorch: {torch.__version__}")

Device: cuda
GPU: NVIDIA GeForce GTX 1070 Ti
CUDA: 12.1, PyTorch: 2.5.1+cu121


1070 Ti, 8GB VRAM. Not a lot for 3D volumes. Patch training, batch size 1, fp16 are all going to be necessary.